<a href="https://colab.research.google.com/github/VictorMagadi/Flame-boss/blob/main/csv3model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from enum import Enum
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Any

# ============================================================================
# CORE CASH FLOW FORECASTING SYSTEM COMPONENTS
# ============================================================================

class TransactionType(Enum):
    SALES_REVENUE = "Sales Revenue"
    SERVICE_REVENUE = "Service Revenue"
    INTEREST_INCOME = "Interest Income"
    OTHER_INCOME = "Other Income"

    SALARIES_WAGES = "Salaries & Wages"
    RENT_LEASE = "Rent & Lease Payments"
    UTILITIES = "Utilities"
    MARKETING_ADVERTISING = "Marketing & Advertising"
    OFFICE_SUPPLIES = "Office Supplies"
    TRAVEL_ENTERTAINMENT = "Travel & Entertainment"
    INSURANCE = "Insurance"
    MAINTENANCE_REPAIRS = "Maintenance & Repairs"
    BANK_FEES = "Bank Fees"
    OPERATING_EXPENSES = "Other Operating Expenses"

    CAPEX_EQUIPMENT = "Capital Expenditure - Equipment"
    INVESTMENT_PURCHASE = "Investment Purchase"
    ASSET_SALE = "Asset Sale"
    INVESTMENT_INCOME = "Investment Income (Dividends/Gains)"

    LOAN_PROCEEDS = "Loan Proceeds"
    LOAN_REPAYMENT = "Loan Repayment"
    EQUITY_INJECTION = "Equity Injection"
    DIVIDEND_PAYOUT = "Dividend Payout"
    INTEREST_PAYMENT = "Interest Payment"

@dataclass
class FinancialTransaction:
    transaction_id: str
    date: datetime
    transaction_type: TransactionType
    category: 'CashFlowCategory'
    amount: float  # Positive for inflow, negative for outflow
    description: str
    is_recurring: bool = False
    recurrence_period: Optional[str] = None  # e.g., 'monthly', 'quarterly', 'annual'

    def __post_init__(self):
        if self.amount == 0:
            raise ValueError("Transaction amount cannot be zero.")

class CashFlowCategory(Enum):
    OPERATING = "Operating Activities"
    INVESTING = "Investing Activities"
    FINANCING = "Financing Activities"

    @staticmethod
    def from_transaction_type(ttype: TransactionType) -> 'CashFlowCategory':
        if ttype in [TransactionType.SALES_REVENUE, TransactionType.SERVICE_REVENUE, TransactionType.INTEREST_INCOME, TransactionType.OTHER_INCOME,
                     TransactionType.SALARIES_WAGES, TransactionType.RENT_LEASE, TransactionType.UTILITIES,
                     TransactionType.MARKETING_ADVERTISING, TransactionType.OFFICE_SUPPLIES,
                     TransactionType.TRAVEL_ENTERTAINMENT, TransactionType.INSURANCE,
                     TransactionType.MAINTENANCE_REPAIRS, TransactionType.BANK_FEES, TransactionType.OPERATING_EXPENSES]:
            return CashFlowCategory.OPERATING
        elif ttype in [TransactionType.CAPEX_EQUIPMENT, TransactionType.INVESTMENT_PURCHASE, TransactionType.ASSET_SALE, TransactionType.INVESTMENT_INCOME]:
            return CashFlowCategory.INVESTING
        elif ttype in [TransactionType.LOAN_PROCEEDS, TransactionType.LOAN_REPAYMENT, TransactionType.EQUITY_INJECTION, TransactionType.DIVIDEND_PAYOUT, TransactionType.INTEREST_PAYMENT]:
            return CashFlowCategory.FINANCING
        else:
            raise ValueError(f"Unknown TransactionType for category mapping: {ttype}")


@dataclass
class CashFlowForecastingSystem:
    company_name: str
    initial_cash: float
    transactions: List[FinancialTransaction] = field(default_factory=list)
    forecast_horizon_months: int = 12

    def add_transaction(self, transaction: FinancialTransaction):
        self.transactions.append(transaction)
        self.transactions.sort(key=lambda x: x.date)

    def generate_recurring_transactions(self, start_date: datetime, end_date: datetime):
        generated_count = 0
        new_transactions = []
        for transaction in self.transactions:
            if transaction.is_recurring and transaction.recurrence_period:
                current_date = transaction.date
                while current_date < start_date:
                    if transaction.recurrence_period == 'monthly':
                        current_date = current_date + timedelta(days=30) # Approximate month
                    elif transaction.recurrence_period == 'quarterly':
                        current_date = current_date + timedelta(days=91)
                    elif transaction.recurrence_period == 'annual':
                        current_date = current_date + timedelta(days=365)
                    else:
                        break # Unsupported recurrence

                while current_date <= end_date:
                    if current_date > transaction.date: # Only generate for future periods
                        new_transaction = FinancialTransaction(
                            transaction_id=f"{transaction.transaction_id}-{current_date.strftime('%Y%m%d')}",
                            date=current_date,
                            transaction_type=transaction.transaction_type,
                            category=transaction.category,
                            amount=transaction.amount,
                            description=f"Recurring: {transaction.description}",
                            is_recurring=False, # Generated instances are not recurring themselves
                            recurrence_period=None
                        )
                        new_transactions.append(new_transaction)
                        generated_count += 1

                    if transaction.recurrence_period == 'monthly':
                        current_date = current_date + timedelta(days=30) # Approximate month
                    elif transaction.recurrence_period == 'quarterly':
                        current_date = current_date + timedelta(days=91)
                    elif transaction.recurrence_period == 'annual':
                        current_date = current_date + timedelta(days=365)
                    else:
                        break # Unsupported recurrence
        self.transactions.extend(new_transactions)
        self.transactions.sort(key=lambda x: x.date)
        print(f"Generated {generated_count} recurring transactions.")

class CashFlowAnalyzer:
    def __init__(self, system: CashFlowForecastingSystem):
        self.system = system

    def calculate_monthly_cash_flow(self, months: Optional[int] = None) -> pd.DataFrame:
        if not self.system.transactions:
            return pd.DataFrame()

        all_dates = sorted([t.date for t in self.system.transactions])
        if not all_dates:
            return pd.DataFrame()

        start_date = all_dates[0].replace(day=1)
        end_date = all_dates[-1].replace(day=1) + timedelta(days=31) # Go to next month to ensure all transactions are covered

        if months:
            end_date = start_date + pd.DateOffset(months=months) - timedelta(days=1)

        # Generate recurring transactions up to the end of the forecast horizon
        self.system.generate_recurring_transactions(start_date, end_date)

        # Filter transactions within the relevant period
        filtered_transactions = [t for t in self.system.transactions if start_date <= t.date <= end_date]

        if not filtered_transactions:
            return pd.DataFrame()

        # Create a DataFrame from transactions
        df = pd.DataFrame([{ 'date': t.date, 'amount': t.amount, 'category': t.category.value, 'type': t.transaction_type.value } for t in filtered_transactions])
        df['month_year'] = df['date'].dt.to_period('M')

        # Calculate inflows and outflows
        inflows = df[df['amount'] > 0].groupby('month_year')['amount'].sum().reindex(pd.period_range(start_date, end_date, freq='M'), fill_value=0)
        outflows = df[df['amount'] < 0].groupby('month_year')['amount'].sum().reindex(pd.period_range(start_date, end_date, freq='M'), fill_value=0)

        monthly_cash_flow = pd.DataFrame({
            'Inflows': inflows,
            'Outflows': outflows
        })
        monthly_cash_flow['Net_Cash_Flow'] = monthly_cash_flow['Inflows'] + monthly_cash_flow['Outflows']

        # Calculate ending cash balance
        current_cash = self.system.initial_cash
        ending_cash_balances = []

        for net_flow in monthly_cash_flow['Net_Cash_Flow']:
            current_cash += net_flow
            ending_cash_balances.append(current_cash)

        monthly_cash_flow['Ending_Cash'] = ending_cash_balances
        monthly_cash_flow['Month_Name'] = monthly_cash_flow.index.strftime('%b %Y')
        monthly_cash_flow = monthly_cash_flow.reset_index(drop=True)

        return monthly_cash_flow

    def calculate_key_metrics(self, monthly_df: pd.DataFrame) -> Dict[str, Any]:
        if monthly_df.empty:
            return {
                "Total Inflows": 0.0,
                "Total Outflows": 0.0,
                "Net Cash Flow": 0.0,
                "Average Monthly Net Cash Flow": 0.0,
                "Ending Cash Balance": self.system.initial_cash,
                "Months of Runway": 0,
                "Cash Flow Volatility (Std Dev)": 0.0
            }

        total_inflows = monthly_df['Inflows'].sum()
        total_outflows = monthly_df['Outflows'].sum()
        net_cash_flow = monthly_df['Net_Cash_Flow'].sum()
        avg_monthly_net_cash_flow = monthly_df['Net_Cash_Flow'].mean()
        ending_cash_balance = monthly_df['Ending_Cash'].iloc[-1]

        # Months of runway: how many months until cash runs out if avg_monthly_net_cash_flow is negative
        months_of_runway = "N/A"
        if avg_monthly_net_cash_flow < 0 and ending_cash_balance > 0:
            months_of_runway = abs(ending_cash_balance / avg_monthly_net_cash_flow)

        cash_flow_volatility = monthly_df['Net_Cash_Flow'].std()

        return {
            "Total Inflows": total_inflows,
            "Total Outflows": total_outflows,
            "Net Cash Flow": net_cash_flow,
            "Average Monthly Net Cash Flow": avg_monthly_net_cash_flow,
            "Ending Cash Balance": ending_cash_balance,
            "Months of Runway": months_of_runway,
            "Cash Flow Volatility (Std Dev)": cash_flow_volatility
        }

    def get_category_breakdown(self, df: pd.DataFrame, flow_type: str) -> pd.Series:
        # Ensure 'date' is datetime for dt accessor
        if not df.empty and not pd.api.types.is_datetime64_any_dtype(df['date']):
            df['date'] = pd.to_datetime(df['date'])

        if flow_type == 'inflow':
            category_df = df[df['amount'] > 0]
        elif flow_type == 'outflow':
            category_df = df[df['amount'] < 0]
        else:
            raise ValueError("flow_type must be 'inflow' or 'outflow'")

        return category_df.groupby('category')['amount'].sum().sort_values(ascending=False)


class CashFlowVisualizer:
    def __init__(self, analyzer: CashFlowAnalyzer):
        self.analyzer = analyzer

    def plot_cash_flow_dashboard(self, monthly_df: pd.DataFrame, metrics: Dict[str, Any]):
        if monthly_df.empty:
            print("No data to plot.")
            return

        fig, axes = plt.subplots(3, 2, figsize=(18, 18))
        fig.suptitle(f"Cash Flow Dashboard for {self.analyzer.system.company_name}", fontsize=20)

        # Plot 1: Monthly Net Cash Flow Trend
        axes[0, 0].plot(monthly_df['Month_Name'], monthly_df['Net_Cash_Flow'], marker='o', linestyle='-', color='skyblue')
        axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=0.8)
        axes[0, 0].set_title('Monthly Net Cash Flow Trend')
        axes[0, 0].set_ylabel('Net Cash Flow ($)')
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].grid(True)

        # Plot 2: Ending Cash Balance Trend
        axes[0, 1].plot(monthly_df['Month_Name'], monthly_df['Ending_Cash'], marker='o', linestyle='-', color='lightgreen')
        axes[0, 1].set_title('Ending Cash Balance Trend')
        axes[0, 1].set_ylabel('Cash Balance ($)')
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True)

        # Plot 3: Monthly Inflows vs Outflows (Bar Chart)
        bar_width = 0.35
        r1 = np.arange(len(monthly_df['Month_Name']))
        r2 = [x + bar_width for x in r1]
        axes[1, 0].bar(r1, monthly_df['Inflows'], color='b', width=bar_width, label='Inflows')
        axes[1, 0].bar(r2, abs(monthly_df['Outflows']), color='r', width=bar_width, label='Outflows')
        axes[1, 0].set_title('Monthly Inflows vs Outflows')
        axes[1, 0].set_ylabel('Amount ($)')
        axes[1, 0].set_xticks([r + bar_width/2 for r in range(len(monthly_df['Month_Name']))])
        axes[1, 0].set_xticklabels(monthly_df['Month_Name'], rotation=45)
        axes[1, 0].legend()
        axes[1, 0].grid(axis='y')

        # Plot 4: Key Metrics (Text Box)
        metrics_text = """Total Inflows: ${:,.2f}
Total Outflows: ${:,.2f}
Net Cash Flow: ${:,.2f}
Avg. Monthly Net CF: ${:,.2f}
Ending Cash Balance: ${:,.2f}
Months of Runway: {}
CF Volatility (Std Dev): ${:,.2f}
""".format(
            metrics['Total Inflows'],
            metrics['Total Outflows'],
            metrics['Net Cash Flow'],
            metrics['Average Monthly Net Cash Flow'],
            metrics['Ending Cash Balance'],
            round(metrics['Months of Runway'], 2) if isinstance(metrics['Months of Runway'], float) else metrics['Months of Runway'],
            metrics['Cash Flow Volatility (Std Dev)']
        )
        axes[1, 1].text(0.05, 0.95, metrics_text, transform=axes[1, 1].transAxes, fontsize=12, verticalalignment='top',
                        bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))
        axes[1, 1].set_title('Key Performance Indicators')
        axes[1, 1].axis('off')

        # Plot 5 & 6: Category Breakdown (Pie Charts)
        # For this, we need a flat list of all transactions within the plotted period
        all_transactions_df = pd.DataFrame([{ 'date': t.date, 'amount': t.amount, 'category': t.category.value } for t in self.analyzer.system.transactions])
        all_transactions_df = all_transactions_df[all_transactions_df['date'].isin(monthly_df.index)] # Filter to plotted months

        inflow_breakdown = self.analyzer.get_category_breakdown(all_transactions_df, 'inflow')
        outflow_breakdown = self.analyzer.get_category_breakdown(all_transactions_df, 'outflow')

        if not inflow_breakdown.empty:
            axes[2, 0].pie(inflow_breakdown, labels=inflow_breakdown.index, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
            axes[2, 0].set_title('Inflow Categories')
            axes[2, 0].axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
        else:
            axes[2, 0].set_title('No Inflow Data')
            axes[2, 0].text(0.5, 0.5, 'No Inflows', horizontalalignment='center', verticalalignment='center', transform=axes[2, 0].transAxes)
            axes[2, 0].axis('off')

        if not outflow_breakdown.empty:
            # Make outflows positive for pie chart percentage calculation
            outflow_breakdown_abs = abs(outflow_breakdown)
            axes[2, 1].pie(outflow_breakdown_abs, labels=outflow_breakdown_abs.index, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
            axes[2, 1].set_title('Outflow Categories')
            axes[2, 1].axis('equal')
        else:
            axes[2, 1].set_title('No Outflow Data')
            axes[2, 1].text(0.5, 0.5, 'No Outflows', horizontalalignment='center', verticalalignment='center', transform=axes[2, 1].transAxes)
            axes[2, 1].axis('off')

        plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to prevent overlap
        plt.show()


class CashFlowOptimizer:
    def __init__(self, analyzer: CashFlowAnalyzer):
        self.analyzer = analyzer

    def get_optimization_suggestions(self) -> List[str]:
        suggestions = []
        monthly_df = self.analyzer.calculate_monthly_cash_flow()

        if monthly_df.empty:
            suggestions.append("No transactions to analyze for optimization.")
            return suggestions

        metrics = self.analyzer.calculate_key_metrics(monthly_df)

        # Suggestion 1: Improve Net Cash Flow if negative
        if metrics['Net Cash Flow'] < 0:
            suggestions.append(f"Warning: Your overall net cash flow is negative (${metrics['Net Cash Flow']:,.2f}). Focus on increasing inflows or decreasing outflows.")

        # Suggestion 2: Identify largest outflows for cost reduction
        # Need to get transaction data for this
        all_transactions_df = pd.DataFrame([{ 'date': t.date, 'amount': t.amount, 'category': t.category.value } for t in self.analyzer.system.transactions])
        if not all_transactions_df.empty:
            outflow_breakdown = self.analyzer.get_category_breakdown(all_transactions_df, 'outflow')
            if not outflow_breakdown.empty:
                largest_outflow_category = outflow_breakdown.index[0]
                largest_outflow_amount = abs(outflow_breakdown.iloc[0])
                suggestions.append(f"Your largest outflow category is '{largest_outflow_category}' (${largest_outflow_amount:,.2f}). Consider strategies to reduce costs in this area.")

        # Suggestion 3: Capitalize on largest inflows
        if not all_transactions_df.empty:
            inflow_breakdown = self.analyzer.get_category_breakdown(all_transactions_df, 'inflow')
            if not inflow_breakdown.empty:
                largest_inflow_category = inflow_breakdown.index[0]
                largest_inflow_amount = inflow_breakdown.iloc[0]
                suggestions.append(f"Your largest inflow category is '{largest_inflow_category}' (${largest_inflow_amount:,.2f}). Explore ways to further leverage and grow this revenue stream.")

        # Suggestion 4: Address low cash runway
        if isinstance(metrics['Months of Runway'], float) and metrics['Months of Runway'] < 3:
            suggestions.append(f"Critical: Your cash runway is only {round(metrics['Months of Runway'], 2)} months. Implement immediate measures to boost cash reserves or cut expenses.")

        # Suggestion 5: Recurring expenses vs. revenues
        recurring_inflows = sum(t.amount for t in self.analyzer.system.transactions if t.is_recurring and t.amount > 0)
        recurring_outflows = sum(abs(t.amount) for t in self.analyzer.system.transactions if t.is_recurring and t.amount < 0)

        if recurring_inflows < recurring_outflows:
            suggestions.append(f"Your recurring outflows (${recurring_outflows:,.2f}) exceed your recurring inflows (${recurring_inflows:,.2f}). Seek to convert more revenue streams into recurring ones or reduce recurring costs.")

        if not suggestions:
            suggestions.append("Cash flow appears healthy. Continue monitoring and look for minor efficiency improvements.")

        return suggestions

In [ ]:
"""
================================================================================
BUSINESS CASH FLOW FORECASTING SYSTEM - WITH EXCEL/CSV IMPORT
================================================================================
Enhanced version that can import transactions from Excel or CSV files.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Any
from enum import Enum
import random
import os

# [Keep all the existing Enum and DataClass definitions from the previous code]
# [TransactionType, CashFlowCategory, FinancialTransaction classes remain the same]

# ============================================================================
# NEW: FILE IMPORT FUNCTIONALITY
# ============================================================================

class TransactionImporter:
    """
    Import transactions from various file formats
    """

    @staticmethod
    def from_csv(filepath: str, date_format: str = '%Y-%m-%d') -> List[FinancialTransaction]:
        """
        Import transactions from CSV file

        Expected CSV columns:
        - transaction_id (optional, will generate if not provided)
        - date (required)
        - transaction_type (required - must match TransactionType enum values)
        - category (required - must match CashFlowCategory enum values)
        - amount (required)
        - description (required)
        - is_recurring (optional, default False)
        - recurrence_period (optional)
        """
        print(f"\nImporting transactions from CSV: {filepath}")

        try:
            df = pd.read_csv(filepath)
            print(f"Found {len(df)} rows in CSV file")

            # Validate required columns
            required_columns = ['date', 'transaction_type', 'category', 'amount', 'description']
            missing_columns = [col for col in required_columns if col not in df.columns]

            if missing_columns:
                raise ValueError(f"Missing required columns: {missing_columns}")

            transactions = []
            transaction_counter = 1000

            for idx, row in df.iterrows():
                try:
                    # Parse date
                    date = datetime.strptime(row['date'], date_format)

                    # Map transaction type string to enum
                    transaction_type = TransactionImporter._map_transaction_type(row['transaction_type'])

                    # Map category string to enum
                    category = TransactionImporter._map_category(row['category'])

                    # Get amount
                    amount = float(row['amount'])

                    # Optional fields
                    is_recurring = bool(row.get('is_recurring', False))
                    recurrence_period = row.get('recurrence_period', None)

                    # Generate transaction ID if not provided
                    if 'transaction_id' in df.columns and pd.notna(row['transaction_id']):
                        transaction_id = str(row['transaction_id'])
                    else:
                        transaction_id = f"IMP{transaction_counter + idx}"

                    # Create transaction
                    transaction = FinancialTransaction(
                        transaction_id=transaction_id,
                        date=date,
                        transaction_type=transaction_type,
                        category=category,
                        amount=amount,
                        description=row['description'],
                        is_recurring=is_recurring,
                        recurrence_period=recurrence_period
                    )

                    transactions.append(transaction)

                except Exception as e:
                    print(f"Error processing row {idx}: {e}")
                    continue

            print(f"Successfully imported {len(transactions)} transactions")
            return transactions

        except Exception as e:
            print(f"Error importing CSV: {e}")
            return []

    @staticmethod
    def from_excel(filepath: str, sheet_name: str = 0, date_format: str = '%Y-%m-%d') -> List[FinancialTransaction]:
        """
        Import transactions from Excel file

        Expected Excel columns (same as CSV):
        - transaction_id (optional)
        - date (required)
        - transaction_type (required)
        - category (required)
        - amount (required)
        - description (required)
        - is_recurring (optional)
        - recurrence_period (optional)
        """
        print(f"\nImporting transactions from Excel: {filepath}")

        try:
            df = pd.read_excel(filepath, sheet_name=sheet_name)
            print(f"Found {len(df)} rows in Excel file")

            # Use same logic as CSV import
            return TransactionImporter.from_dataframe(df, date_format)

        except Exception as e:
            print(f"Error importing Excel: {e}")
            return []

    @staticmethod
    def from_dataframe(df: pd.DataFrame, date_format: str = '%Y-%m-%d') -> List[FinancialTransaction]:
        """
        Import transactions from pandas DataFrame
        """
        # Validate required columns
        required_columns = ['date', 'transaction_type', 'category', 'amount', 'description']
        missing_columns = [col for col in required_columns if col not in df.columns]

        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        transactions = []
        transaction_counter = 1000

        for idx, row in df.iterrows():
            try:
                # Handle date - could be string or datetime
                if isinstance(row['date'], str):
                    date = datetime.strptime(row['date'], date_format)
                else:
                    date = row['date']  # Assume it's already datetime

                # Map transaction type string to enum
                transaction_type = TransactionImporter._map_transaction_type(row['transaction_type'])

                # Map category string to enum
                category = TransactionImporter._map_category(row['category'])

                # Get amount
                amount = float(row['amount'])

                # Optional fields
                is_recurring = bool(row.get('is_recurring', False))
                recurrence_period = row.get('recurrence_period', None)

                # Generate transaction ID if not provided
                if 'transaction_id' in df.columns and pd.notna(row.get('transaction_id')):
                    transaction_id = str(row['transaction_id'])
                else:
                    transaction_id = f"IMP{transaction_counter + idx}"

                # Create transaction
                transaction = FinancialTransaction(
                    transaction_id=transaction_id,
                    date=date,
                    transaction_type=transaction_type,
                    category=category,
                    amount=amount,
                    description=row['description'],
                    is_recurring=is_recurring,
                    recurrence_period=recurrence_period
                )

                transactions.append(transaction)

            except Exception as e:
                print(f"Error processing row {idx}: {e}")
                continue

        print(f"Successfully imported {len(transactions)} transactions")
        return transactions

    @staticmethod
    def _map_transaction_type(type_str: str) -> TransactionType:
        """Map string to TransactionType enum"""
        # Try direct match first
        type_str = str(type_str).strip()

        for t in TransactionType:
            if t.value.lower() == type_str.lower() or t.name.lower() == type_str.lower():
                return t

        # Try partial match
        for t in TransactionType:
            if type_str.lower() in t.value.lower() or type_str.lower() in t.name.lower():
                print(f"Warning: Partial match '{type_str}' -> '{t.value}'")
                return t

        # Default to OPERATING_EXPENSES if no match
        print(f"Warning: Unknown transaction type '{type_str}', defaulting to OPERATING_EXPENSES")
        return TransactionType.OPERATING_EXPENSES

    @staticmethod
    def _map_category(category_str: str) -> CashFlowCategory:
        """Map string to CashFlowCategory enum"""
        category_str = str(category_str).strip().lower()

        for c in CashFlowCategory:
            if c.value.lower() == category_str or c.name.lower() == category_str:
                return c

        # Try to infer from keywords
        if 'operating' in category_str or 'op' in category_str:
            return CashFlowCategory.OPERATING
        elif 'invest' in category_str:
            return CashFlowCategory.INVESTING
        elif 'financ' in category_str:
            return CashFlowCategory.FINANCING

        # Default to OPERATING
        print(f"Warning: Unknown category '{category_str}', defaulting to OPERATING")
        return CashFlowCategory.OPERATING

    @staticmethod
    def create_template(filepath: str, file_type: str = 'csv'):
        """
        Create a template file with the required columns
        """
        template_data = {
            'transaction_id': ['TXN001', 'TXN002', 'TXN003'],
            'date': ['2024-01-15', '2024-01-31', '2024-02-01'],
            'transaction_type': ['SALES_REVENUE', 'SALARIES_WAGES', 'RENT_LEASE'],
            'category': ['OPERATING', 'OPERATING', 'OPERATING'],
            'amount': [50000.00, -25000.00, -8000.00],
            'description': ['Monthly sales', 'Payroll', 'Office rent'],
            'is_recurring': [True, True, True],
            'recurrence_period': ['monthly', 'monthly', 'monthly']
        }

        df = pd.DataFrame(template_data)

        if file_type.lower() == 'csv':
            df.to_csv(filepath, index=False)
            print(f"CSV template created: {filepath}")
        elif file_type.lower() == 'excel':
            df.to_excel(filepath, index=False)
            print(f"Excel template created: {filepath}")
        else:
            print(f"Unsupported file type: {file_type}")

# ============================================================================
# MODIFIED: Enhanced CashFlowForecastingSystem with import methods
# ============================================================================

class EnhancedCashFlowForecastingSystem(CashFlowForecastingSystem):
    """
    Enhanced version with file import capabilities
    """

    def import_from_csv(self, filepath: str, date_format: str = '%Y-%m-%d'):
        """Import transactions from CSV file"""
        transactions = TransactionImporter.from_csv(filepath, date_format)
        self.transactions.extend(transactions)
        print(f"Added {len(transactions)} transactions to system")
        return len(transactions)

    def import_from_excel(self, filepath: str, sheet_name: str = 0, date_format: str = '%Y-%m-%d'):
        """Import transactions from Excel file"""
        transactions = TransactionImporter.from_excel(filepath, sheet_name, date_format)
        self.transactions.extend(transactions)
        print(f"Added {len(transactions)} transactions to system")
        return len(transactions)

    def import_from_dataframe(self, df: pd.DataFrame, date_format: str = '%Y-%m-%d'):
        """Import transactions from pandas DataFrame"""
        transactions = TransactionImporter.from_dataframe(df, date_format)
        self.transactions.extend(transactions)
        print(f"Added {len(transactions)} transactions to system")
        return len(transactions)

    def export_to_csv(self, filepath: str = 'exported_transactions.csv'):
        """Export all transactions to CSV"""
        data = []
        for t in self.transactions:
            data.append({
                'transaction_id': t.transaction_id,
                'date': t.date.strftime('%Y-%m-%d'),
                'transaction_type': t.transaction_type.name,
                'category': t.category.name,
                'amount': t.amount,
                'description': t.description,
                'is_recurring': t.is_recurring,
                'recurrence_period': t.recurrence_period
            })

        df = pd.DataFrame(data)
        df.to_csv(filepath, index=False)
        print(f"Exported {len(data)} transactions to {filepath}")

    def export_to_excel(self, filepath: str = 'exported_transactions.xlsx'):
        """Export all transactions to Excel"""
        data = []
        for t in self.transactions:
            data.append({
                'transaction_id': t.transaction_id,
                'date': t.date.strftime('%Y-%m-%d'),
                'transaction_type': t.transaction_type.name,
                'category': t.category.name,
                'amount': t.amount,
                'description': t.description,
                'is_recurring': t.is_recurring,
                'recurrence_period': t.recurrence_period
            })

        df = pd.DataFrame(data)
        df.to_excel(filepath, index=False)
        print(f"Exported {len(data)} transactions to {filepath}")

# ============================================================================
# EXAMPLE USAGE WITH EXTERNAL FILES
# ============================================================================

def demonstrate_file_import():
    """
    Demonstrate how to use the system with external files
    """

    print("\n" + "=" * 80)
    print("DEMONSTRATION: IMPORTING TRANSACTIONS FROM EXTERNAL FILES")
    print("=" * 80)

    # 1. Create a template file first (so you know the format)
    print("\n1. CREATING TEMPLATE FILE")
    print("-" * 50)
    TransactionImporter.create_template('transaction_template.csv', 'csv')
    TransactionImporter.create_template('transaction_template.xlsx', 'excel')

    # 2. Initialize the enhanced system
    print("\n2. INITIALIZING ENHANCED SYSTEM")
    print("-" * 50)
    system = EnhancedCashFlowForecastingSystem(
        company_name="Import Demo Company",
        initial_cash=500000
    )

    # 3. Option A: Import from CSV
    print("\n3A. IMPORTING FROM CSV")
    print("-" * 50)
    csv_file = 'sample_transactions.csv'

    # Create a sample CSV file
    sample_data = pd.DataFrame({
        'transaction_id': ['S001', 'S002', 'S003', 'S004', 'S005'],
        'date': ['2024-01-15', '2024-01-20', '2024-01-25', '2024-02-01', '2024-02-15'],
        'transaction_type': ['SALES_REVENUE', 'SALARIES_WAGES', 'UTILITIES', 'RENT_LEASE', 'MARKETING_ADVERTISING'],
        'category': ['OPERATING', 'OPERATING', 'OPERATING', 'OPERATING', 'OPERATING'],
        'amount': [75000, -28000, -3500, -8500, -4200],
        'description': ['Product sales', 'Bi-weekly payroll', 'Electricity bill', 'February rent', 'Google Ads'],
        'is_recurring': [True, True, True, True, True],
        'recurrence_period': ['monthly', 'bi-weekly', 'monthly', 'monthly', 'monthly']
    })
    sample_data.to_csv(csv_file, index=False)

    count = system.import_from_csv(csv_file)
    print(f"Imported {count} transactions from CSV")

    # 4. Option B: Import from Excel
    print("\n3B. IMPORTING FROM EXCEL")
    print("-" * 50)
    excel_file = 'sample_transactions.xlsx'

    # Add some investing and financing transactions
    excel_data = pd.DataFrame({
        'transaction_id': ['I001', 'I002', 'F001', 'F002'],
        'date': ['2024-01-10', '2024-01-30', '2024-01-05', '2024-01-31'],
        'transaction_type': ['CAPEX_EQUIPMENT', 'ASSET_SALE', 'LOAN_PROCEEDS', 'INTEREST_PAYMENT'],
        'category': ['INVESTING', 'INVESTING', 'FINANCING', 'FINANCING'],
        'amount': [-35000, 12000, 100000, -1200],
        'description': ['New machinery', 'Sale of old equipment', 'Business loan', 'Monthly interest'],
        'is_recurring': [False, False, False, True],
        'recurrence_period': [None, None, None, 'monthly']
    })
    excel_data.to_excel(excel_file, index=False)

    count = system.import_from_excel(excel_file)
    print(f"Imported {count} transactions from Excel")

    # 5. Analyze the imported data
    print("\n4. ANALYZING IMPORTED DATA")
    print("-" * 50)
    analyzer = CashFlowAnalyzer(system)
    monthly_df = analyzer.calculate_monthly_cash_flow(months=3)  # Only 3 months of data

    print("\nMonthly Cash Flow from Imported Data:")
    print(monthly_df.to_string(index=False))

    # 6. Calculate metrics
    metrics = analyzer.calculate_key_metrics(monthly_df)

    print("\nKey Metrics from Imported Data:")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key:30s}: ${value:,.2f}")
        else:
            print(f"{key:30s}: {value}")

    # 7. Export the transactions (in case you want to save them)
    print("\n5. EXPORTING TRANSACTIONS")
    print("-" * 50)
    system.export_to_csv('my_exported_data.csv')
    system.export_to_excel('my_exported_data.xlsx')

    # 8. Create visualizations
    print("\n6. CREATING VISUALIZATIONS")
    print("-" * 50)
    visualizer = CashFlowVisualizer(analyzer)
    visualizer.plot_cash_flow_dashboard(monthly_df, metrics)

    return system, analyzer, monthly_df, metrics

# ============================================================================
# PRACTICAL USAGE EXAMPLES
# ============================================================================

def practical_usage_scenarios():
    """
    Show different ways to use the import functionality
    """

    print("\n" + "=" * 80)
    print("PRACTICAL USAGE SCENARIOS")
    print("=" * 80)

    # Scenario 1: Quick analysis from CSV
    print("\nSCENARIO 1: Quick Analysis from CSV")
    print("-" * 50)

    # Create a CSV file with your transactions
    your_data = pd.DataFrame({
        'date': ['2024-01-01', '2024-01-15', '2024-01-31', '2024-02-01'],
        'transaction_type': ['SALES_REVENUE', 'SALARIES_WAGES', 'RENT_LEASE', 'SALES_REVENUE'],
        'category': ['OPERATING', 'OPERATING', 'OPERATING', 'OPERATING'],
        'amount': [100000, -45000, -12000, 85000],
        'description': ['January sales', 'Monthly payroll', 'Office rent', 'February sales']
    })
    your_data.to_csv('my_actual_data.csv', index=False)

    # Load and analyze
    system = EnhancedCashFlowForecastingSystem("My Company", 200000)
    count = system.import_from_csv('my_actual_data.csv')

    analyzer = CashFlowAnalyzer(system)
    monthly_df = analyzer.calculate_monthly_cash_flow()

    print(f"\nAnalyzed {count} transactions")
    if not monthly_df.empty:
        print("\nResults:")
        print(monthly_df[['Month_Name', 'Inflows', 'Outflows', 'Net_Cash_Flow', 'Ending_Cash']])

    # Scenario 2: Batch processing multiple files
    print("\nSCENARIO 2: Batch Processing Multiple Files")
    print("-" * 50)

    # Create multiple files
    for month in range(1, 4):
        file_name = f'transactions_month_{month}.csv'
        month_data = pd.DataFrame({
            'date': [f'2024-{month:02d}-15', f'2024-{month:02d}-30'],
            'transaction_type': ['SALES_REVENUE', 'OPERATING_EXPENSES'],
            'category': ['OPERATING', 'OPERATING'],
            'amount': [80000 + month*5000, -30000 - month*2000],
            'description': [f'Month {month} sales', f'Month {month} expenses']
        })
        month_data.to_csv(file_name, index=False)
        print(f"Created {file_name}")

    # Load all files
    batch_system = EnhancedCashFlowForecastingSystem("Batch Company", 300000)
    total_count = 0

    for month in range(1, 4):
        file_name = f'transactions_month_{month}.csv'
        count = batch_system.import_from_csv(file_name)
        total_count += count

    print(f"\nLoaded {total_count} transactions from 3 files")

    # Scenario 3: Using with accounting software export
    print("\nSCENARIO 3: QuickBooks/Xero Export Format")
    print("-" * 50)

    # Simulate accounting software export format
    accounting_export = pd.DataFrame({
        'Transaction Date': ['01/15/2024', '01/31/2024', '02/01/2024', '02/15/2024'],
        'Account': ['Sales Revenue', 'Payroll Expense', 'Rent Expense', 'Marketing Expense'],
        'Amount': [95000, -38000, -9000, -5500],
        'Description': ['Monthly sales', 'Employee payroll', 'Office lease', 'Facebook ads'],
        'Transaction Type': ['Income', 'Expense', 'Expense', 'Expense']
    })
    accounting_export.to_csv('quickbooks_export.csv', index=False)

    # Custom import function for specific format
    def import_quickbooks_format(filepath):
        df = pd.read_csv(filepath)

        # Map QuickBooks format to our format
        mapped_data = []
        for _, row in df.iterrows():
            # Parse date
            date = datetime.strptime(row['Transaction Date'], '%m/%d/%Y')

            # Map account to transaction type
            account = row['Account'].lower()
            if 'sales' in account or 'revenue' in account:
                trans_type = TransactionType.SALES_REVENUE
            elif 'payroll' in account or 'salary' in account:
                trans_type = TransactionType.SALARIES_WAGES
            elif 'rent' in account:
                trans_type = TransactionType.RENT_LEASE
            elif 'marketing' in account or 'ad' in account:
                trans_type = TransactionType.MARKETING_ADVERTISING
            else:
                trans_type = TransactionType.OPERATING_EXPENSES

            mapped_data.append({
                'date': date,
                'transaction_type': trans_type.name,
                'category': CashFlowCategory.OPERATING.name,
                'amount': row['Amount'],
                'description': row['Description']
            })

        return pd.DataFrame(mapped_data)

    # Import the converted data
    mapped_df = import_quickbooks_format('quickbooks_export.csv')
    qb_system = EnhancedCashFlowForecastingSystem("QuickBooks Company", 150000)
    qb_system.import_from_dataframe(mapped_df, date_format='%Y-%m-%d')

    print("Successfully imported QuickBooks format data")

# ============================================================================
# MAIN EXECUTION WITH FILE IMPORT OPTIONS
# ============================================================================

def main_with_import():
    """
    Main function with file import options
    """

    print("\n" + "=" * 80)
    print("BUSINESS CASH FLOW FORECASTING SYSTEM - WITH FILE IMPORT")
    print("=" * 80)

    while True:
        print("\n" + "=" * 50)
        print("MENU OPTIONS")
        print("=" * 50)
        print("1. Generate sample data (for testing)")
        print("2. Import from CSV file")
        print("3. Import from Excel file")
        print("4. Create template file")
        print("5. Run demonstration with sample files")
        print("6. Exit")

        choice = input("\nEnter your choice (1-6): ").strip()

        if choice == '1':
            # Use original main() function
            from main import main as original_main
            original_main()

        elif choice == '2':
            filepath = input("Enter CSV file path: ").strip()
            if os.path.exists(filepath):
                system = EnhancedCashFlowForecastingSystem("Imported Company", 250000)
                count = system.import_from_csv(filepath)

                if count > 0:
                    analyzer = CashFlowAnalyzer(system)
                    monthly_df = analyzer.calculate_monthly_cash_flow()

                    if not monthly_df.empty:
                        metrics = analyzer.calculate_key_metrics(monthly_df)
                        visualizer = CashFlowVisualizer(analyzer)
                        visualizer.plot_cash_flow_dashboard(monthly_df, metrics)

                        print("\nAnalysis complete! Check the generated dashboard.")
            else:
                print(f"File not found: {filepath}")

        elif choice == '3':
            filepath = input("Enter Excel file path: ").strip()
            if os.path.exists(filepath):
                sheet_name = input("Enter sheet name (press Enter for first sheet): ").strip()
                sheet = sheet_name if sheet_name else 0

                system = EnhancedCashFlowForecastingSystem("Imported Company", 250000)
                count = system.import_from_excel(filepath, sheet)

                if count > 0:
                    analyzer = CashFlowAnalyzer(system)
                    monthly_df = analyzer.calculate_monthly_cash_flow()

                    if not monthly_df.empty:
                        metrics = analyzer.calculate_key_metrics(monthly_df)
                        visualizer = CashFlowVisualizer(analyzer)
                        visualizer.plot_cash_flow_dashboard(monthly_df, metrics)

                        print("\nAnalysis complete! Check the generated dashboard.")
            else:
                print(f"File not found: {filepath}")

        elif choice == '4':
            file_type = input("Enter file type (csv or excel): ").strip().lower()
            if file_type in ['csv', 'excel']:
                filename = input(f"Enter filename (default: template.{file_type}): ").strip()
                if not filename:
                    filename = f"template.{file_type}"
                TransactionImporter.create_template(filename, file_type)
            else:
                print("Invalid file type. Please enter 'csv' or 'excel'")

        elif choice == '5':
            demonstrate_file_import()
            practical_usage_scenarios()

        elif choice == '6':
            print("\nExiting...")
            break

        else:
            print("Invalid choice. Please try again.")

# ============================================================================
# REQUIRED COLUMNS FOR IMPORT FILES
# ============================================================================

"""
REQUIRED COLUMNS FOR CSV/EXCEL FILES:
---------------------------------------
1. date: Transaction date (format: YYYY-MM-DD)
2. transaction_type: Must match TransactionType enum names or values
   Examples: SALES_REVENUE, SALARIES_WAGES, RENT_LEASE, etc.
3. category: Must match CashFlowCategory enum names or values
   Examples: OPERATING, INVESTING, FINANCING
4. amount: Transaction amount (positive for inflows, negative for outflows)
5. description: Transaction description

OPTIONAL COLUMNS:
-----------------
6. transaction_id: Unique identifier (will be auto-generated if not provided)
7. is_recurring: True/False
8. recurrence_period: 'monthly', 'quarterly', 'annual', etc.

TRANSACTION TYPE MAPPING:
-------------------------
The system will try to match your transaction types to the enum values.
You can use either the enum name or the display value:
- 'SALES_REVENUE' or 'Sales Revenue'
- 'SALARIES_WAGES' or 'Salaries & Wages'
- etc.

CATEGORY MAPPING:
-----------------
- 'OPERATING' or 'Operating Activities'
- 'INVESTING' or 'Investing Activities'
- 'FINANCING' or 'Financing Activities'
"""

# ============================================================================
# EXAMPLE: HOW TO USE WITH YOUR OWN DATA
# ============================================================================

def example_with_your_data():
    """
    Example showing how to use the system with your own data
    """

    # Step 1: Load your data (from CSV, Excel, or database)
    print("Step 1: Load your data")
    your_data = pd.read_csv('your_transactions.csv')  # or pd.read_excel()

    # Step 2: Initialize the system
    system = EnhancedCashFlowForecastingSystem("Your Company", 100000)

    # Step 3: Import your data
    system.import_from_dataframe(your_data)

    # Step 4: Run analysis
    analyzer = CashFlowAnalyzer(system)
    monthly_df = analyzer.calculate_monthly_cash_flow()
    metrics = analyzer.calculate_key_metrics(monthly_df)

    # Step 5: View results
    print("\nMonthly Cash Flow:")
    print(monthly_df)

    print("\nKey Metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value}")

    # Step 6: Generate visualizations
    visualizer = CashFlowVisualizer(analyzer)
    visualizer.plot_cash_flow_dashboard(monthly_df, metrics)

    # Step 7: Get optimization suggestions
    optimizer = CashFlowOptimizer(analyzer)
    suggestions = optimizer.get_optimization_suggestions()

    print("\nOptimization Suggestions:")
    for suggestion in suggestions:
        print(f"  {suggestion}")

if __name__ == "__main__":
    # Run the interactive menu
    main_with_import()

    # Or directly run the demonstration
    # demonstrate_file_import()
    # practical_usage_scenarios()


BUSINESS CASH FLOW FORECASTING SYSTEM - WITH FILE IMPORT

MENU OPTIONS
1. Generate sample data (for testing)
2. Import from CSV file
3. Import from Excel file
4. Create template file
5. Run demonstration with sample files
6. Exit
